# Multi-Model Research with E5 Embeddings and Optuna

This notebook explores NLI classification using pre-computed `multilingual-e5-small` embeddings.
We use Optuna for hyperparameter optimization across several models: XGBoost, LightGBM, and SVM.

In [ ]:
import json
from pathlib import Path

import joblib
import mlflow
import numpy as np
import pandas as pd
import optuna
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from helpers.data import load_processed_data, load_sample_submission
from helpers.metrics import compute_all_metrics
from helpers.mlflow import log_common_context, log_metrics, log_model_artifacts, start_notebook_run
from helpers.submission import build_submission, save_submission

## Load Data and Embeddings

In [ ]:
PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
EMBEDDINGS_DIR = Path('../embeddings/e5')

train_df, val_df, test_df = load_processed_data(PROCESSED_DIR)
sample_submission = load_sample_submission(RAW_DIR / 'sample_submission.csv')

def load_embs(prefix):
    p = np.load(EMBEDDINGS_DIR / f'{prefix}_premise.npy')
    h = np.load(EMBEDDINGS_DIR / f'{prefix}_hypothesis.npy')
    return p, h

train_p, train_h = load_embs('train')
val_p, val_h = load_embs('val')
test_p, test_h = load_embs('test')

print(f'Data and embeddings loaded.')

## Feature Engineering

We combine premise and hypothesis embeddings using concatenation, absolute difference, and element-wise product.

In [ ]:
def prepare_features(p, h):
    return np.hstack([
        p, 
        h, 
        np.abs(p - h), 
        p * h
    ])

x_train = prepare_features(train_p, train_h)
x_val = prepare_features(val_p, val_h)
x_test = prepare_features(test_p, test_h)

y_train = train_df['label'].astype(int)
y_val = val_df['label'].astype(int)

print(f'Feature shape: {x_train.shape}')

## Optuna Optimization

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42,
        'eval_metric': 'mlogloss',
        'n_jobs': -1,
    }
    model = XGBClassifier(**params)
    model.fit(x_train, y_train)
    return model.score(x_val, y_val)

def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'random_state': 42,
        'verbosity': -1,
        'n_jobs': -1,
    }
    model = LGBMClassifier(**params)
    model.fit(x_train, y_train)
    return model.score(x_val, y_val)

def objective_svm(trial):
    params = {
        'C': trial.suggest_float('C', 0.1, 10.0, log=True),
        'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf']),
        'probability': True,
        'random_state': 42,
    }
    model = SVC(**params)
    model.fit(x_train, y_train)
    return model.score(x_val, y_val)

## Run Experiments

In [ ]:
MODELS = {
    'xgboost': objective_xgb,
    'lightgbm': objective_lgbm,
    'svm': objective_svm,
}

results = []

for model_name, objective in MODELS.items():
    print(f'Optimizing {model_name}...')
    study = optuna.create_study(direction='maximize')
    # Limiting trials for research purposes
    study.optimize(objective, n_trials=10 if model_name != 'svm' else 5, show_progress_bar=True, n_jobs=1)
    
    best_params = study.best_params
    print(f'Best params for {model_name}: {best_params}')
    
    # Train final model with best params
    if model_name == 'xgboost':
        model = XGBClassifier(**best_params, random_state=42, eval_metric='mlogloss', n_jobs=-1)
    elif model_name == 'lightgbm':
        model = LGBMClassifier(**best_params, random_state=42, verbosity=-1, n_jobs=-1)
    else:
        model = SVC(**best_params, random_state=42, probability=True)
        
    model.fit(x_train, y_train)
    val_preds = model.predict(x_val)
    metrics = compute_all_metrics(val_df, val_preds)
    test_preds = model.predict(x_test)
    
    run_name = f'e5_optuna_{model_name}'
    submission = build_submission(sample_submission, test_preds)
    submission_path = Path('../submissions') / f'{run_name}.csv'
    save_submission(submission, submission_path)
    
    model_path = Path('../submissions') / f'{run_name}_model.joblib'
    joblib.dump(model, model_path)
    
    with start_notebook_run(
        run_name,
        tags={'model_family': model_name, 'features': 'e5_embeddings', 'optimized': 'optuna'}
    ):
        mlflow.log_params(best_params)
        log_metrics(metrics)
        log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
        mlflow.log_artifact(str(submission_path), artifact_path='submissions')
        log_model_artifacts(
            artifacts={'model.joblib': model_path},
            metadata={'model_name': model_name, 'best_params': best_params},
            artifact_path='model'
        )
    
    results.append({
        'model': model_name,
        'accuracy': metrics['accuracy'],
        'f1_macro': metrics['f1_macro'],
        'best_params': best_params
    })

pd.DataFrame(results).sort_values('f1_macro', ascending=False)

In [ ]:
pd.DataFrame(results).sort_values('f1_macro', ascending=False)

## Language-Aware Stacking with Full Artifact Logging

This advanced stacking implementation:
1. Validates class order consistency across all models.
2. Appends encoded Language IDs to model probabilities.
3. Uses an XGBoost meta-model tuned with Optuna.
4. Logs all base models, the meta-model, and the encoder for full reproducibility.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

print("Preparing robust language-aware stacking...")

# 1. Encode Languages
le = LabelEncoder()
train_lang = le.fit_transform(train_df['lang_abv']).reshape(-1, 1)
val_lang = le.transform(val_df['lang_abv']).reshape(-1, 1)
test_lang = le.transform(test_df['lang_abv']).reshape(-1, 1)
joblib.dump(le, 'label_encoder.joblib')

EXPECTED_CLASSES = np.array([0, 1, 2])

# 2. Generate Out-of-Fold (OOF) Predictions for Base Models
def get_oof_predictions(model_name, params, x, y, x_v, x_t):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_probs = np.zeros((len(x), 3))
    val_probs = np.zeros((len(x_v), 3))
    test_probs = np.zeros((len(x_t), 3))
    
    for train_idx, holdout_idx in skf.split(x, y):
        if model_name == 'xgboost':
            m = XGBClassifier(**params, random_state=42, eval_metric='mlogloss', n_jobs=-1)
        elif model_name == 'lightgbm':
            m = LGBMClassifier(**params, random_state=42, verbosity=-1, n_jobs=-1)
        else:
            m = SVC(**params, random_state=42, probability=True)
            
        m.fit(x[train_idx], y.iloc[train_idx])
        
        # Safety Check: Class order
        if not np.array_equal(m.classes_, EXPECTED_CLASSES):
            raise ValueError(f"Model {model_name} has unexpected class order: {m.classes_}")
            
        oof_probs[holdout_idx] = m.predict_proba(x[holdout_idx])
        val_probs += m.predict_proba(x_v) / 5
        test_probs += m.predict_proba(x_t) / 5
        
    return oof_probs, val_probs, test_probs

best_params_dict = {res['model']: res['best_params'] for res in results}
meta_train_list = []
meta_val_list = []
meta_test_list = []

for m_name in ['xgboost', 'lightgbm', 'svm']:
    print(f"Generating OOF for {m_name}...")
    o, v, t = get_oof_predictions(m_name, best_params_dict[m_name], x_train, y_train, x_val, x_test)
    meta_train_list.append(o)
    meta_val_list.append(v)
    meta_test_list.append(t)

# Combine model probs with language ID
x_meta_train = np.hstack(meta_train_list + [train_lang])
x_meta_val = np.hstack(meta_val_list + [val_lang])
x_meta_test = np.hstack(meta_test_list + [test_lang])

# 3. Tune XGBoost Meta-Model with Optuna
def objective_meta_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'random_state': 42,
        'eval_metric': 'mlogloss',
        'n_jobs': -1,
    }
    model = XGBClassifier(**params)
    model.fit(x_meta_train, y_train)
    return model.score(x_meta_val, y_val)

print("Optimizing Meta-XGBoost...")
study_meta = optuna.create_study(direction='maximize')
study_meta.optimize(objective_meta_xgb, n_trials=20, show_progress_bar=True)

best_meta_params = study_meta.best_params
print(f"Best meta-params: {best_meta_params}")

# 4. Final Train (Meta-Model AND Base Models on 100% data)
print("Final training on all data...")
final_meta_model = XGBClassifier(**best_meta_params, random_state=42, eval_metric='mlogloss', n_jobs=-1)
final_meta_model.fit(x_meta_train, y_train)

base_model_paths = {}
for m_name in ['xgboost', 'lightgbm', 'svm']:
    if m_name == 'xgboost':
        m = XGBClassifier(**best_params_dict[m_name], random_state=42, eval_metric='mlogloss', n_jobs=-1)
    elif m_name == 'lightgbm':
        m = LGBMClassifier(**best_params_dict[m_name], random_state=42, verbosity=-1, n_jobs=-1)
    else:
        m = SVC(**best_params_dict[m_name], random_state=42, probability=True)
    
    m.fit(x_train, y_train)
    p = f"base_{m_name}.joblib"
    joblib.dump(m, p)
    base_model_paths[f"base_{m_name}.joblib"] = p

meta_model_path = "meta_model.joblib"
joblib.dump(final_meta_model, meta_model_path)

# 5. Evaluate and Log
val_preds = final_meta_model.predict(x_meta_val)
metrics = compute_all_metrics(val_df, val_preds)
test_preds = final_meta_model.predict(x_meta_test)

run_name = 'e5_language_aware_stacking_robust'
save_submission(build_submission(sample_submission, test_preds), Path('../submissions') / f'{run_name}.csv')

with start_notebook_run(
    run_name,
    tags={'model_family': 'stacking', 'features': 'probs_and_lang', 'meta_model': 'xgboost'}
):
    mlflow.log_params(best_meta_params)
    log_metrics(metrics)
    log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
    
    # Log EVERYTHING: meta-model, all base models, and the encoder
    all_artifacts = {"meta_model.joblib": meta_model_path, "label_encoder.joblib": "label_encoder.joblib"}
    all_artifacts.update(base_model_paths)
    
    log_model_artifacts(
        artifacts=all_artifacts,
        metadata={'best_meta_params': best_meta_params, 'base_models_used': list(best_params_dict.keys())},
        artifact_path='model'
    )

print("Robust language-aware stacking completed.")
pd.Series(metrics)

## Performance Analysis & Visualizations

We analyze the model's performance across different languages and visualize the confusion matrix to identify common misclassifications.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
print("Generating visualizations...")
# 1. Prepare data for per-language metrics
results_df = val_df.copy()
results_df['pred'] = val_preds
languages = sorted(results_df['language'].unique())
metrics_by_lang = []
for lang in languages:
    lang_data = results_df[results_df['language'] == lang]
    m = compute_all_metrics(lang_data, lang_data['pred'])
    metrics_by_lang.append({
        'Language': lang,
        'Accuracy': m['accuracy'],
        'Precision': m['precision_macro'],
        'Recall': m['recall_macro'],
        'F1-macro': m['f1_macro']
    })
viz_df = pd.DataFrame(metrics_by_lang).melt(id_vars='Language', var_name='Metric', value_name='Score')
# 2. Prepare data for per-class metrics
precision, recall, f1, _ = precision_recall_fscore_support(y_val, val_preds, labels=[0, 1, 2])
class_names = ['entailment', 'neutral', 'contradiction']
per_class_metrics = []
for i, name in enumerate(class_names):
    per_class_metrics.append({'Class': name, 'Metric': 'Precision', 'Score': precision[i]})
    per_class_metrics.append({'Class': name, 'Metric': 'Recall', 'Score': recall[i]})
    per_class_metrics.append({'Class': name, 'Metric': 'F1-score', 'Score': f1[i]})
class_viz_df = pd.DataFrame(per_class_metrics)
# 3. Plotting
sns.set_theme(style="whitegrid", palette="muted")
# Збільшуємо висоту до 28, щоб вмістити 4 графіки
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(16, 28))
# --- Plot 1: Per-language metrics ---
sns.barplot(data=viz_df, x='Language', y='Score', hue='Metric', ax=ax1, width=0.97, gap=0)
for container in ax1.containers:
    ax1.bar_label(container, fmt='%.2f', padding=-40, fontsize=14, rotation=90, label_type='edge', fontweight='bold', color='white')
ax1.set_title('Performance Metrics by Language', fontsize=20, fontweight='bold', pad=20)
ax1.set_ylim(0, 1.05)
ax1.legend(title='Metric', bbox_to_anchor=(1, 1), loc='upper left')
ax1.set_xlabel('Language', fontsize=14)
plt.setp(ax1.get_xticklabels(), rotation=45)
ax1.set_ylabel('Score', fontsize=14)
# --- Plot 2: Language distribution in training data ---
sns.countplot(data=train_df, x='language', ax=ax2, order=languages, palette="viridis")
ax2.set_title('Language Distribution in Training Data', fontsize=20, fontweight='bold', pad=20)
ax2.set_xlabel('Language', fontsize=14)
ax2.set_ylabel('Number of Examples', fontsize=14)
plt.setp(ax2.get_xticklabels(), rotation=45)
for container in ax2.containers:
    ax2.bar_label(container, fontsize=12, fontweight='bold', padding=3)
# --- Plot 3: Performance Metrics by Class ---
sns.barplot(data=class_viz_df, x='Class', y='Score', hue='Metric', ax=ax3, palette="magma")
for container in ax3.containers:
    ax3.bar_label(container, fmt='%.2f', padding=-40, fontsize=14, rotation=0, label_type='edge', fontweight='bold', color='white')
ax3.set_title('Performance Metrics by Class (Global)', fontsize=20, fontweight='bold', pad=20)
ax3.set_ylim(0, 1.05)
ax3.set_xlabel('Class', fontsize=14)
ax3.set_ylabel('Score', fontsize=14)
# --- Plot 4: Confusion Matrix ---
cm = confusion_matrix(y_val, val_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', ax=ax4,
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={"size": 16, "weight": "bold"})
ax4.set_title('Overall Confusion Matrix', fontsize=20, fontweight='bold', pad=20)
ax4.set_xlabel('Predicted Label', fontsize=14)
ax4.set_ylabel('True Label', fontsize=14)
plt.tight_layout()
plt.show()
print("Visualizations generated successfully.")